# BinaryTask: CatBoost и XGBoost

Обе реализации обучаются на одинаковых split и feature columns. Для быстрого интеграционного прогона каждая использует Optuna с одним фиксированным trial. Сравниваются ROC AUC и распределения score; точное совпадение не ожидается, потому что алгоритмы различаются.

In [1]:
from pathlib import Path
from time import perf_counter

import polars as pl
from IPython.display import display

from avatar.automl import BinaryTask, BinaryTaskConfig

WORKSPACE = Path.cwd().parent.resolve()
TRAIN_PATH = WORKSPACE / 'data/binary/train'
VALID_PATH = WORKSPACE / 'data/binary/valid'
TEST_PATH = WORKSPACE / 'data/binary/test'
CAT_COLS = [f'cat_feature_{index}' for index in range(1, 6)]
NUM_COLS = [f'num_feature_{index}' for index in range(1, 6)]
SEARCH_SPACES = {
    'catboost': {'iterations': [16], 'depth': [2], 'l2_leaf_reg': [1.0], 'bagging_temperature': [0.25], 'learning_rate': [0.1]},
    'xgboost': {'n_estimators': [16], 'max_depth': [2], 'lambda': [1.0], 'alpha': [0.0], 'subsample': [0.8], 'colsample_bytree': [0.8], 'eta': [0.1]},
}


In [2]:
rows = []
for engine in ('catboost', 'xgboost'):
    config = BinaryTaskConfig(environment={}, 
        env_type='local', backend='boosting', engine=engine, device='gpu',
        target_column='target', client_id_column='epk_id', report_month_column='report_month',
        group_column='group', treatment_column='treatment', inverse_treatment=True, model_scope='product',
        categorical_columns=CAT_COLS, numerical_columns=NUM_COLS, hidden_state_columns=['seq_hidden_state'],
        hyperopt=True, optimization_metric='roc_auc', n_trials=1, random_state=42, verbose=False,
        search_space=SEARCH_SPACES[engine],
        output_dir=WORKSPACE / f'outputs/notebook_tests/engines/{engine}',
    )
    task = BinaryTask(config)
    started = perf_counter()
    training = task.train(TRAIN_PATH, VALID_PATH)
    train_seconds = perf_counter() - started
    scores = task.predict(TEST_PATH)
    evaluation = task.evaluate(TEST_PATH, scores)
    rows.append({
        'engine': engine, 'validation_roc_auc': training.validation_metrics['product'],
        'test_roc_auc': evaluation.metrics['roc_auc'], 'score_mean': scores.scores['score'].mean(),
        'score_std': scores.scores['score'].std(), 'score_p05': scores.scores['score'].quantile(0.05),
        'score_p50': scores.scores['score'].quantile(0.50), 'score_p95': scores.scores['score'].quantile(0.95),
        'train_seconds': train_seconds,
    })

engine_summary = pl.DataFrame(rows)
display(engine_summary)


2026-08-17 13:41:26,871 INFO run_id=56763c22b66d47aaa94197fe34bb422f Starting action=train task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


<workspace>\fmlib-main\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-17 13:41:28,768] A new study created in memory with name: no-name-3fc42c3a-be9e-4d0b-a05d-6a58ea13a475


[I 2026-08-17 13:41:31,792] Trial 0 finished with value: 0.7242333449101337 and parameters: {'iterations': 16, 'depth': 2, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.25, 'learning_rate': 0.1}. Best is trial 0 with value: 0.7242333449101337.


2026-08-17 13:41:31,797 INFO run_id=56763c22b66d47aaa94197fe34bb422f Finished action=train


2026-08-17 13:41:31,802 INFO run_id=b4a496941b0d4c329e7a4c58704ffb5c Starting action=predict task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


...

2026-08-17 13:41:39,629 INFO run_id=9c825c6a1990415b8d046be42d8f8e0e Starting action=predict task_config=BinaryTaskConfig backend=boosting engine=xgboost env_type=local


2026-08-17 13:41:39,991 INFO run_id=9c825c6a1990415b8d046be42d8f8e0e Finished action=predict


2026-08-17 13:41:40,004 INFO run_id=a3f3d680a9d14e0db1cac8088fecbcee Starting action=evaluate task_config=BinaryTaskConfig backend=boosting engine=xgboost env_type=local


2026-08-17 13:41:41,329 INFO run_id=a3f3d680a9d14e0db1cac8088fecbcee Finished action=evaluate


engine,validation_roc_auc,test_roc_auc,score_mean,score_std,score_p05,score_p50,score_p95,train_seconds
str,f64,f64,f64,f64,f64,f64,f64,f64
"""catboost""",0.724233,0.723482,0.333599,0.112153,0.183555,0.311262,0.539948,4.930701
"""xgboost""",0.724993,0.72402,0.306208,0.1001,0.174201,0.284793,0.496981,3.754096


Таблица предназначена для визуальной проверки: оба engine должны давать ROC AUC заметно выше случайного уровня и score в сопоставимом диапазоне. Различия допустимы и ожидаемы.